# IMS Bearing Dataset Audit

This notebook is the first inventory pass for the IMS bearing dataset. It does not train a model. It checks what data is present, whether the experiment files are usable, and whether the observed files match the IMS documentation.

Expected raw layout after extraction:

- `data/raw/IMS/IMS/1st_test/`
- `data/raw/IMS/IMS/2nd_test/`
- `data/raw/IMS/IMS/3rd_test/`

If only `.rar` archives are present, this notebook will still inventory the archives and mark the measurement-file audit as pending extraction.

In [1]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
import math
import os
import re
from typing import Iterable

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DIR = PROJECT_ROOT / "data" / "raw"
IMS_DIR = RAW_DIR / "IMS" / "IMS"

EXPECTED_EXPERIMENTS = {
    "1st_test": {
        "expected_files": 2156,
        "expected_channels": 8,
        "expected_measurements_per_file": 20480,
        "recording_duration": "2003-10-22 12:06:24 to 2003-11-25 23:39:56",
        "recording_interval": "Every 10 minutes, except first 43 files every 5 minutes",
        "failed_bearing": "Bearing 3 inner race defect; Bearing 4 roller element defect",
    },
    "2nd_test": {
        "expected_files": 984,
        "expected_channels": 4,
        "expected_measurements_per_file": 20480,
        "recording_duration": "2004-02-12 10:32:39 to 2004-02-19 06:22:39",
        "recording_interval": "Every 10 minutes",
        "failed_bearing": "Bearing 1 outer race failure",
    },
    "3rd_test": {
        "expected_files": 4448,
        "expected_channels": 4,
        "expected_measurements_per_file": 20480,
        "recording_duration": "2004-03-04 09:27:46 to 2004-04-04 19:01:57",
        "recording_interval": "Every 10 minutes",
        "failed_bearing": "Bearing 3 outer race failure",
    },
}

AUDIT_FILE_LIMIT = os.getenv("IMS_AUDIT_MAX_FILES")
AUDIT_FILE_LIMIT = int(AUDIT_FILE_LIMIT) if AUDIT_FILE_LIMIT else None

EXPECTED_ORDER = list(EXPECTED_EXPERIMENTS)
PROJECT_ROOT, RAW_DIR, IMS_DIR

(WindowsPath('C:/Users/Balsem/Desktop/GMAO/ai-service'),
 WindowsPath('C:/Users/Balsem/Desktop/GMAO/ai-service/data/raw'),
 WindowsPath('C:/Users/Balsem/Desktop/GMAO/ai-service/data/raw/IMS/IMS'))

## 1. Raw Dataset and Archive Inventory

In [2]:
def format_bytes(size: int | float | None) -> str:
    if size is None or pd.isna(size):
        return ""
    units = ["B", "KB", "MB", "GB", "TB"]
    value = float(size)
    for unit in units:
        if abs(value) < 1024 or unit == units[-1]:
            return f"{value:,.2f} {unit}"
        value /= 1024
    return f"{value:,.2f} TB"


def safe_rel(path: Path) -> str:
    try:
        return path.resolve().relative_to(PROJECT_ROOT).as_posix()
    except ValueError:
        return path.resolve().as_posix()


raw_items = []
if RAW_DIR.exists():
    for path in sorted(RAW_DIR.rglob("*")):
        if "__MACOSX" in path.parts:
            continue
        if path.name.startswith("._"):
            continue
        if path.is_file():
            raw_items.append(
                {
                    "path": safe_rel(path),
                    "suffix": path.suffix.lower(),
                    "bytes": path.stat().st_size,
                    "size": format_bytes(path.stat().st_size),
                    "last_modified": pd.Timestamp(path.stat().st_mtime, unit="s"),
                }
            )

raw_inventory = pd.DataFrame(raw_items)
raw_inventory

,path,suffix,bytes,size,last_modified
0,data/raw/.gitkeep,,1,1.00 B,2026-08-30 00:40:05.266632794
1,data/raw/IMS/IMS/1st_test/2003.10.22.12.06.24,.24,1148716,1.10 MB,2004-05-12 19:02:08.312500000
2,data/raw/IMS/IMS/1st_test/2003.10.22.12.09.13,.13,1149141,1.10 MB,2004-05-12 19:02:08.703125000
3,data/raw/IMS/IMS/1st_test/2003.10.22.12.14.13,.13,1149766,1.10 MB,2004-05-12 19:02:09.125000000
4,data/raw/IMS/IMS/1st_test/2003.10.22.12.19.13,.13,1149168,1.10 MB,2004-05-12 19:02:09.875000000
...,...,...,...,...,...
9465,data/raw/IMS/IMS/3rd_test/txt/2004.04.18.02.32.55,.55,553022,540.06 KB,2004-04-19 11:18:50.000000000
9466,data/raw/IMS/IMS/3rd_test/txt/2004.04.18.02.42.55,.55,512044,500.04 KB,2004-04-19 11:18:50.000000000
9467,data/raw/IMS/IMS/3rd_test.rar,.rar,609047134,580.83 MB,2017-07-18 22:33:55.000000000
9468,data/raw/IMS/IMS/Readme Document for IMS Beari...,.pdf,400443,391.06 KB,2012-10-10 04:26:16.000000000


In [3]:
archive_inventory = raw_inventory[raw_inventory["suffix"].isin([".zip", ".rar", ".7z", ".tar", ".gz"])] if not raw_inventory.empty else pd.DataFrame()
archive_inventory

,path,suffix,bytes,size,last_modified
2157,data/raw/IMS/IMS/1st_test.rar,.rar,366567310,349.59 MB,2007-09-18 07:44:22.000000000
3142,data/raw/IMS/IMS/2nd_test.rar,.rar,85581092,81.62 MB,2007-09-18 13:54:12.000000000
9467,data/raw/IMS/IMS/3rd_test.rar,.rar,609047134,580.83 MB,2017-07-18 22:33:55.000000000
9469,data/raw/IMS.zip,.zip,1061902801,"1,012.71 MB",2026-08-30 01:38:07.477511883


## 2. Expected Experiment Metadata

In [4]:
expected_metadata = pd.DataFrame.from_dict(EXPECTED_EXPERIMENTS, orient="index").reset_index(names="experiment")
expected_metadata

,experiment,expected_files,expected_channels,expected_measurements_per_file,recording_duration,recording_interval,failed_bearing
0,1st_test,2156,8,20480,2003-10-22 12:06:24 to 2003-11-25 23:39:56,"Every 10 minutes, except first 43 files every ...",Bearing 3 inner race defect; Bearing 4 roller ...
1,2nd_test,984,4,20480,2004-02-12 10:32:39 to 2004-02-19 06:22:39,Every 10 minutes,Bearing 1 outer race failure
2,3rd_test,4448,4,20480,2004-03-04 09:27:46 to 2004-04-04 19:01:57,Every 10 minutes,Bearing 3 outer race failure


## 3. Measurement File Discovery

In [5]:
def find_experiment_dir(experiment: str) -> Path | None:
    candidates = [
        IMS_DIR / experiment,
        RAW_DIR / "IMS" / experiment,
        RAW_DIR / experiment,
    ]
    for candidate in candidates:
        if candidate.is_dir():
            return candidate
    return None


def find_experiment_archive(experiment: str) -> Path | None:
    archive_names = [f"{experiment}.rar", f"{experiment}.zip", f"{experiment}.7z"]
    search_dirs = [IMS_DIR, RAW_DIR / "IMS", RAW_DIR]
    for folder in search_dirs:
        if not folder.is_dir():
            continue
        for archive_name in archive_names:
            candidate = folder / archive_name
            if candidate.is_file():
                return candidate
    return None


def measurement_files(experiment_dir: Path | None) -> list[Path]:
    if experiment_dir is None or not experiment_dir.exists():
        return []
    ignored_suffixes = {".zip", ".rar", ".7z", ".pdf", ".md"}
    files = []
    for path in experiment_dir.rglob("*"):
        if not path.is_file():
            continue
        if "__MACOSX" in path.parts or path.name.startswith((".", "._")):
            continue
        if path.suffix.lower() in ignored_suffixes:
            continue
        files.append(path)
    return sorted(files, key=lambda p: p.name)


discovery_rows = []
for experiment, expected in EXPECTED_EXPERIMENTS.items():
    experiment_dir = find_experiment_dir(experiment)
    archive = find_experiment_archive(experiment)
    files = measurement_files(experiment_dir)
    bytes_on_disk = sum(path.stat().st_size for path in files)
    discovery_rows.append(
        {
            "experiment": experiment,
            "experiment_dir": safe_rel(experiment_dir) if experiment_dir else None,
            "archive": safe_rel(archive) if archive else None,
            "archive_size": format_bytes(archive.stat().st_size) if archive else None,
            "measurement_files_found": len(files),
            "expected_files": expected["expected_files"],
            "missing_file_count_vs_expected": expected["expected_files"] - len(files),
            "extra_file_count_vs_expected": max(len(files) - expected["expected_files"], 0),
            "extracted_measurement_size": format_bytes(bytes_on_disk),
            "failed_bearing": expected["failed_bearing"],
        }
    )

experiment_discovery = pd.DataFrame(discovery_rows)

known_experiment_dirs = {
    Path(path).name for path in experiment_discovery["experiment_dir"].dropna()
}
candidate_experiment_dirs = []
for folder in [IMS_DIR, RAW_DIR / "IMS", RAW_DIR]:
    if folder.is_dir():
        candidate_experiment_dirs.extend(
            path for path in folder.glob("*_test")
            if path.is_dir() and path.name not in EXPECTED_EXPERIMENTS
        )

unexpected_experiment_dirs = pd.DataFrame(
    {"unexpected_experiment_dir": safe_rel(path)} for path in sorted(set(candidate_experiment_dirs))
)
dataset_inventory_summary = pd.DataFrame(
    [
        {
            "expected_experiments": len(EXPECTED_EXPERIMENTS),
            "expected_experiments_with_archives": int(experiment_discovery["archive"].notna().sum()),
            "expected_experiments_extracted": int(experiment_discovery["experiment_dir"].notna().sum()),
            "unexpected_experiment_dirs": len(unexpected_experiment_dirs),
        }
    ]
)
dataset_inventory_summary

,expected_experiments,expected_experiments_with_archives,expected_experiments_extracted,unexpected_experiment_dirs
0,3,3,3,0


## 4. File-Level Audit Helpers

In [6]:
TIMESTAMP_FORMATS = [
    "%Y.%m.%d.%H.%M.%S",
    "%Y-%m-%d-%H-%M-%S",
    "%Y_%m_%d_%H_%M_%S",
]


def parse_timestamp_from_name(path: Path):
    candidates = [path.name, path.stem]
    for candidate in dict.fromkeys(candidates):
        for fmt in TIMESTAMP_FORMATS:
            try:
                return pd.Timestamp(pd.to_datetime(candidate, format=fmt))
            except (ValueError, TypeError):
                pass
    loose_match = re.search(r"(\d{4})[._-](\d{2})[._-](\d{2})[._-](\d{2})[._-](\d{2})[._-](\d{2})", path.name)
    if loose_match:
        return pd.Timestamp("-".join(loose_match.groups()[:3]) + " " + ":".join(loose_match.groups()[3:]))
    return pd.NaT


@dataclass
class RunningStats:
    count: int = 0
    mean: float = 0.0
    m2: float = 0.0
    minimum: float = math.inf
    maximum: float = -math.inf

    def update(self, values: np.ndarray) -> None:
        flat = values.astype(float, copy=False).ravel()
        flat = flat[np.isfinite(flat)]
        if flat.size == 0:
            return
        batch_count = int(flat.size)
        batch_mean = float(flat.mean())
        batch_m2 = float(((flat - batch_mean) ** 2).sum())
        delta = batch_mean - self.mean
        total_count = self.count + batch_count
        self.mean += delta * batch_count / total_count
        self.m2 += batch_m2 + delta * delta * self.count * batch_count / total_count
        self.count = total_count
        self.minimum = min(self.minimum, float(flat.min()))
        self.maximum = max(self.maximum, float(flat.max()))

    def as_dict(self) -> dict[str, float | int | None]:
        if self.count == 0:
            return {"value_count": 0, "min": None, "max": None, "mean": None, "std": None}
        return {
            "value_count": self.count,
            "min": self.minimum,
            "max": self.maximum,
            "mean": self.mean,
            "std": math.sqrt(self.m2 / (self.count - 1)) if self.count > 1 else 0.0,
        }


def read_measurement_file(path: Path) -> pd.DataFrame:
    return pd.read_csv(path, sep=r"\s+", header=None, engine="python")


def audit_measurement_file(path: Path) -> tuple[dict, np.ndarray | None]:
    base = {
        "path": safe_rel(path),
        "file_name": path.name,
        "bytes": path.stat().st_size,
        "timestamp": parse_timestamp_from_name(path),
        "status": "ok",
        "error": None,
        "measurements": None,
        "channels": None,
    }
    if path.stat().st_size == 0:
        base["status"] = "empty"
        return base, None
    try:
        frame = read_measurement_file(path)
        if frame.empty:
            base["status"] = "empty"
            return base, None
        numeric = frame.apply(pd.to_numeric, errors="coerce")
        if numeric.isna().any().any():
            base["status"] = "malformed"
            base["error"] = "Non-numeric values found"
            return base, None
        base["measurements"] = int(numeric.shape[0])
        base["channels"] = int(numeric.shape[1])
        return base, numeric.to_numpy(dtype=float)
    except Exception as exc:
        base["status"] = "malformed"
        base["error"] = f"{type(exc).__name__}: {exc}"
        return base, None

## 5. Experiment-Level Audit

In [7]:
def audit_experiment(experiment: str) -> tuple[dict, pd.DataFrame]:
    expected = EXPECTED_EXPERIMENTS[experiment]
    experiment_dir = find_experiment_dir(experiment)
    files = measurement_files(experiment_dir)
    files_to_audit = files[:AUDIT_FILE_LIMIT] if AUDIT_FILE_LIMIT is not None else files
    stats = RunningStats()
    file_rows = []

    for path in files_to_audit:
        row, values = audit_measurement_file(path)
        file_rows.append(row)
        if values is not None:
            stats.update(values)

    files_df = pd.DataFrame(file_rows)
    ok_files = files_df[files_df["status"].eq("ok")] if not files_df.empty else pd.DataFrame()
    timestamps = ok_files["timestamp"].dropna().tolist() if not ok_files.empty else []
    chronological = timestamps == sorted(timestamps) if timestamps else None
    duplicate_timestamps = int(ok_files["timestamp"].duplicated().sum()) if not ok_files.empty and "timestamp" in ok_files else 0
    missing_timestamps = int(files_df["timestamp"].isna().sum()) if not files_df.empty else 0

    summary = {
        "experiment": experiment,
        "experiment_dir": safe_rel(experiment_dir) if experiment_dir else None,
        "files_found": len(files),
        "files_scanned": len(files_to_audit),
        "audit_file_limit": AUDIT_FILE_LIMIT,
        "expected_files": expected["expected_files"],
        "files_missing_vs_expected": expected["expected_files"] - len(files),
        "extra_files_vs_expected": max(len(files) - expected["expected_files"], 0),
        "ok_files": int(files_df["status"].eq("ok").sum()) if not files_df.empty else 0,
        "empty_files": int(files_df["status"].eq("empty").sum()) if not files_df.empty else 0,
        "malformed_files": int(files_df["status"].eq("malformed").sum()) if not files_df.empty else 0,
        "expected_channels": expected["expected_channels"],
        "observed_channel_counts": sorted(ok_files["channels"].dropna().astype(int).unique().tolist()) if not ok_files.empty else [],
        "expected_measurements_per_file": expected["expected_measurements_per_file"],
        "observed_measurement_counts": sorted(ok_files["measurements"].dropna().astype(int).unique().tolist()) if not ok_files.empty else [],
        "first_timestamp": min(timestamps) if timestamps else pd.NaT,
        "last_timestamp": max(timestamps) if timestamps else pd.NaT,
        "chronological_by_filename": chronological,
        "duplicate_timestamps": duplicate_timestamps,
        "missing_or_unparsed_timestamps": missing_timestamps,
        "failed_bearing": expected["failed_bearing"],
        "measurement_bytes": sum(path.stat().st_size for path in files),
        "measurement_size": format_bytes(sum(path.stat().st_size for path in files)),
        **stats.as_dict(),
    }
    return summary, files_df


summaries = []
file_audits: dict[str, pd.DataFrame] = {}
for experiment in EXPECTED_ORDER:
    summary, files_df = audit_experiment(experiment)
    summaries.append(summary)
    file_audits[experiment] = files_df

audit_summary = pd.DataFrame(summaries)
audit_summary

,experiment,experiment_dir,files_found,files_scanned,audit_file_limit,expected_files,files_missing_vs_expected,extra_files_vs_expected,ok_files,empty_files,...,duplicate_timestamps,missing_or_unparsed_timestamps,failed_bearing,measurement_bytes,measurement_size,value_count,min,max,mean,std
0,1st_test,data/raw/IMS/IMS/1st_test,2156,2,2,2156,0,0,2,0,...,0,0,Bearing 3 inner race defect; Bearing 4 roller ...,2477767237,2.31 GB,327680,-0.784,0.701,-0.092843,0.079687
1,2nd_test,data/raw/IMS/IMS/2nd_test,984,2,2,984,0,0,2,0,...,0,0,Bearing 1 outer race failure,544618480,519.39 MB,163840,-0.911,1.023,-0.007356,0.085040
2,3rd_test,data/raw/IMS/IMS/3rd_test,6324,2,2,4448,-1876,1876,2,0,...,0,0,Bearing 3 outer race failure,3502648779,3.26 GB,163840,-0.569,0.547,-0.005169,0.074906


## 6. Missing, Empty, or Malformed Files

In [8]:
problem_rows = []
for experiment, files_df in file_audits.items():
    summary_row = audit_summary[audit_summary["experiment"].eq(experiment)].iloc[0]
    if files_df.empty:
        status = "not_scanned" if summary_row["files_found"] > 0 else "not_extracted"
        error = "Measurement files were found, but none were scanned because IMS_AUDIT_MAX_FILES limited the audit." if status == "not_scanned" else "No measurement files found. Extract the experiment archive first."
        problem_rows.append(
            {
                "experiment": experiment,
                "status": status,
                "path": None,
                "error": error,
            }
        )
        continue
    bad = files_df[~files_df["status"].eq("ok")]
    for _, row in bad.iterrows():
        problem_rows.append(
            {
                "experiment": experiment,
                "status": row["status"],
                "path": row["path"],
                "error": row["error"],
            }
        )

problem_files = pd.DataFrame(problem_rows)
problem_files

""


## 7. Timestamp and Chronology Details

In [9]:
chronology_rows = []
for experiment, files_df in file_audits.items():
    if files_df.empty:
        continue
    ordered = files_df.sort_values("file_name").reset_index(drop=True)
    parsed = ordered["timestamp"].dropna()
    if parsed.empty:
        continue
    deltas = parsed.sort_values().diff().dropna()
    chronology_rows.append(
        {
            "experiment": experiment,
            "first_timestamp": parsed.min(),
            "last_timestamp": parsed.max(),
            "chronological_by_filename": parsed.tolist() == sorted(parsed.tolist()),
            "minimum_delta": deltas.min() if not deltas.empty else pd.NaT,
            "median_delta": deltas.median() if not deltas.empty else pd.NaT,
            "maximum_delta": deltas.max() if not deltas.empty else pd.NaT,
        }
    )

chronology_summary = pd.DataFrame(chronology_rows)
chronology_summary

,experiment,first_timestamp,last_timestamp,chronological_by_filename,minimum_delta,median_delta,maximum_delta
0,1st_test,2003-10-22 12:06:24,2003-10-22 12:09:13,True,0 days 00:02:49,0 days 00:02:49,0 days 00:02:49
1,2nd_test,2004-02-12 10:32:39,2004-02-12 10:42:39,True,0 days 00:10:00,0 days 00:10:00,0 days 00:10:00
2,3rd_test,2004-03-04 09:27:46,2004-03-04 09:32:46,True,0 days 00:05:00,0 days 00:05:00,0 days 00:05:00


## 8. Dataset Size Summary

In [10]:
raw_bytes = int(raw_inventory["bytes"].sum()) if not raw_inventory.empty else 0
measurement_bytes = int(audit_summary["measurement_bytes"].sum()) if "measurement_bytes" in audit_summary else 0
size_summary = pd.DataFrame(
    [
        {"scope": "raw files currently on disk", "bytes": raw_bytes, "size": format_bytes(raw_bytes)},
        {"scope": "extracted measurement files", "bytes": measurement_bytes, "size": format_bytes(measurement_bytes)},
    ]
)
size_summary

,scope,bytes,size
0,raw files currently on disk,8648533277,8.05 GB
1,extracted measurement files,6525034496,6.08 GB


## 9. Inventory Verdict

Run all cells after extracting the experiment archives. A trustworthy inventory should show:

- 3 experiments discovered.
- Expected file counts: 2,156, 984, and 4,448.
- Expected channels: 8 for the first experiment, 4 for the second and third.
- Expected measurements per file: 20,480.
- No empty or malformed files.
- Parsed timestamps in chronological order.
- Failed bearing notes populated from the IMS documentation.

Do not begin model training until the audit summary and problem-files table are clean or every exception has a documented reason.